# Discussão crítica e demonstração

Consolidação das limitações, uso prático e uma predição de exemplo com aviso legal.

- **Entrada:** `modelos_treinados.joblib` e `metricas_teste.json` (notebook 05)
- **Alvo:** 1 = maligno, 0 = benigno
- **Saída:** texto pronto para o relatório PDF (fase v3) + demo de inferência
- **Próximo:** extrair código para `src/` e entrega FIAP (fase v3)

## 1 - Setup e carregar artefatos

In [1]:
# Célula comum (v2) + imports da demo
from pathlib import Path
import json

import joblib
import pandas as pd

# Encontra a raiz do repositório (notebooks em subpastas ou na raiz)
def encontrar_raiz() -> Path:
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "src" / "db").is_dir():
            return candidato
    raise FileNotFoundError("Raiz do repositório não encontrada.")

RAIZ = encontrar_raiz()
OUTPUT_BASE = RAIZ / "src/db/outputs/breast-cancer-wisconsin-data"

PASTA_ENTRADA_05 = OUTPUT_BASE / "05-avaliacao-metricas"
PASTA_BASE = OUTPUT_BASE / "07-discussao-demo"
PASTA_FIGURAS = PASTA_BASE / "figures"
PASTA_METRICAS = PASTA_BASE / "metrics"
PASTA_ARTEFATOS = PASTA_BASE / "artifacts"

CAMINHO_ARTEFATO_05 = PASTA_ENTRADA_05 / "artifacts" / "modelos_treinados.joblib"
CAMINHO_METRICAS_05 = PASTA_ENTRADA_05 / "metrics" / "metricas_teste.json"

for pasta in (PASTA_FIGURAS, PASTA_METRICAS, PASTA_ARTEFATOS):
    pasta.mkdir(parents=True, exist_ok=True)

In [2]:
# Carrega modelos + métricas do notebook 05 — sem retreinar
if not CAMINHO_ARTEFATO_05.exists():
    raise FileNotFoundError("Execute o notebook 05 primeiro (artefato de modelos).")
if not CAMINHO_METRICAS_05.exists():
    raise FileNotFoundError("Execute o notebook 05 primeiro (metricas_teste.json).")

pacote = joblib.load(CAMINHO_ARTEFATO_05)
modelos_treinados = pacote["modelos_treinados"]
atributos_teste = pacote["atributos_teste"]
rotulo_teste = pacote["rotulo_teste"]
colunas_atributos = pacote["colunas_atributos"]

with open(CAMINHO_METRICAS_05, encoding="utf-8") as f:
    metricas = json.load(f)

nome_melhor = metricas["_melhor_modelo"]
melhor_pipeline = modelos_treinados[nome_melhor]

print(f"Artefato 05: {CAMINHO_ARTEFATO_05}")
print(f"Métricas 05: {CAMINHO_METRICAS_05}")
print(f"Modelos: {list(modelos_treinados.keys())}")
print(f"Melhor modelo: {nome_melhor}")
print(f"Teste: {len(atributos_teste)} amostras | Atributos: {len(colunas_atributos)}")

Artefato 05: E:\_git\fiap\fase1\tech-challenge-fase1\src\db\outputs\breast-cancer-wisconsin-data\05-avaliacao-metricas\artifacts\modelos_treinados.joblib
Métricas 05: E:\_git\fiap\fase1\tech-challenge-fase1\src\db\outputs\breast-cancer-wisconsin-data\05-avaliacao-metricas\metrics\metricas_teste.json
Modelos: ['Regressão Logística', 'Árvore de Decisão', 'SVM', 'Gradient Boosting', 'Random Forest']
Melhor modelo: SVM
Teste: 114 amostras | Atributos: 30


## 2 - Discussão crítica

> Texto abaixo pode ser copiado para o relatório PDF (fase v3, task 09).

### Uso prático

O modelo classifica tumores de mama como **maligno** ou **benigno** a partir de atributos numéricos derivados de imagens de aspiração por agulha fina (FNA). O uso pretendido é **triagem** e **segunda opinião** no fluxo clínico — por exemplo, priorizar casos para revisão ou apoiar a leitura de exames. **Não substitui biópsia**, histopatologia nem o julgamento clínico completo.

### Limitações

- **Dataset pequeno:** Breast Cancer Wisconsin Diagnostic tem ~569 amostras; métricas no holdout podem variar com outro split ou outra coorte.
- **População específica:** dados de um contexto institucional/histórico; generalização para outros hospitais, equipamentos ou populações não está garantida.
- **Features já extraídas:** trabalhamos com 30 atributos numéricos pré-calculados, não com imagens brutas — o pipeline não cobre erro de aquisição ou de segmentação da imagem.
- **Desbalanceamento leve e colinearidade:** classes não estão 50/50 e atributos de tamanho (`radius`, `perimeter`, `area`) são fortemente correlacionados, o que afeta interpretação de importância.

### Tipos de erro e impacto clínico

| Erro | Significado | Impacto |
|------|-------------|--------|
| **Falso negativo (FN)** | Maligno previsto como benigno | **Mais grave:** atraso no diagnóstico e no tratamento |
| **Falso positivo (FP)** | Benigno previsto como maligno | Ansiedade, exames e procedimentos adicionais |

Por isso priorizamos **recall de maligno** na escolha do modelo (notebook 05), aceitando algum custo em falsos positivos.

### Papel do médico

A **palavra final é sempre humana**. O sistema é apoio à decisão: deve ser auditável (métricas, matriz de confusão, SHAP), transparente quanto às limitações e usado sob supervisão profissional. Em caso de dúvida clínica, prevalece a avaliação médica — não a saída do algoritmo.

## 3 - Resumo das métricas (conjunto de teste)

Tabela gerada a partir de `metricas_teste.json` do notebook 05. O melhor modelo foi escolhido pelo contexto clínico (priorizando recall de maligno).

In [3]:
# Tabela de métricas (exclui metadados com prefixo _)
with open(CAMINHO_METRICAS_05, encoding="utf-8") as f:
    metricas = json.load(f)

linhas = {k: v for k, v in metricas.items() if not k.startswith("_")}
tabela_metricas = pd.DataFrame(linhas).T
display(tabela_metricas)

print(f"\nMelhor modelo (notebook 05): {metricas['_melhor_modelo']}")

,acuracia,recall_maligno,f1_maligno,precisao_maligno
Regressão Logística,0.964912,0.928571,0.951220,0.975000
Árvore de Decisão,0.929825,0.904762,0.904762,0.904762
SVM,0.982456,0.976190,0.976190,0.976190
Gradient Boosting,0.964912,0.904762,0.950000,1.000000
Random Forest,0.973684,0.928571,0.962963,1.000000



Melhor modelo (notebook 05): SVM


## 4 - Demo — predição de exemplo

Inferência em **uma amostra do conjunto de teste** com o melhor pipeline. O aviso legal reforça o uso como apoio, não como diagnóstico definitivo.

In [4]:
# Predição de exemplo (1ª linha do teste) + aviso legal
amostra = atributos_teste.iloc[[0]]
rotulo_real = int(rotulo_teste.iloc[0])
pred = melhor_pipeline.predict(amostra)[0]
prob = melhor_pipeline.predict_proba(amostra)[0][1]

print("Modelo:", nome_melhor)
print("Rótulo real:", "maligno" if rotulo_real == 1 else "benigno")
print("Diagnóstico previsto:", "maligno" if pred == 1 else "benigno")
print(f"Probabilidade maligno: {prob:.2%}")
print("\nAVISO: Apoio à decisão. Não substitui diagnóstico médico.")

Modelo: SVM
Rótulo real: benigno
Diagnóstico previsto: benigno
Probabilidade maligno: 0.07%

AVISO: Apoio à decisão. Não substitui diagnóstico médico.


## 5 - Conclusão e próximos passos

### Síntese

Nesta fase analítica (notebooks 01–07):

1. Definimos o problema de classificação binária (maligno vs benigno) e o risco crítico do falso negativo.
2. Exploramos o dataset Wisconsin Diagnostic e preparamos o pipeline (split, escala quando necessário).
3. Treinamos pelo menos dois modelos (Regressão Logística e Árvore de Decisão).
4. Avaliamos no **teste**, escolhemos o melhor pelo contexto clínico e documentamos importância + SHAP.
5. Discutimos limitações, erros e o papel do médico; demos um exemplo de inferência com aviso legal.

### Próximos passos (fase v3)

- Extrair código repetido dos notebooks para módulos em `src/` (treino, avaliação, inferência).
- Consolidar README, relatório PDF e vídeo de demonstração para a entrega FIAP.
- Manter a cadeia: artefatos versionados em `outputs/` / `src/db/outputs/` e notebooks como documentação executável.

**Fase v2 (notebooks) concluída** quando 01–07 executam em ordem e os artefatos de métricas/figuras existem.

## 6 - Validação

In [5]:
# Confirma artefatos da cadeia v2 usados neste notebook
assert CAMINHO_ARTEFATO_05.exists(), "Falta modelos_treinados.joblib (notebook 05)"
assert CAMINHO_METRICAS_05.exists(), "Falta metricas_teste.json (notebook 05)"
assert nome_melhor in modelos_treinados
assert "_melhor_modelo" in metricas
assert len(atributos_teste) > 0

print("Validacao OK")
print("Melhor modelo:", nome_melhor)
print("Métricas disponíveis:", list(linhas.keys()))
print("Amostra demo — pred:", int(pred), "| prob_maligno:", round(float(prob), 4))

Validacao OK
Melhor modelo: SVM
Métricas disponíveis: ['Regressão Logística', 'Árvore de Decisão', 'SVM', 'Gradient Boosting', 'Random Forest']
Amostra demo — pred: 0 | prob_maligno: 0.0007
